# VASP Gemma E2B QLoRA Training (Planner + Refiner)

This notebook trains two adapters on your paired datasets:
- Planner adapter (`planner_e2b_qlora`)
- Refiner adapter (`refiner_e2b_qlora`)

It assumes these files exist in repo:
- `finetune/configs/train_planner_paired_qlora.yaml`
- `finetune/configs/train_refiner_paired_qlora.yaml`
- `finetune/data/planner_paired_train.jsonl`
- `finetune/data/planner_paired_val.jsonl`
- `finetune/data/refiner_paired_train.jsonl`
- `finetune/data/refiner_paired_val.jsonl`


In [2]:
# 1) Check GPU runtime
!nvidia-smi


Mon Apr 20 23:01:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   30C    P0             47W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
# # 2) Clone repo (edit URL)
REPO_URL = "https://github.com/theextraordinary/VASP.git"
REPO_DIR = "/content/VASP"

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repo already exists, pulling latest...")
    %cd {REPO_DIR}
    !git pull
%cd {REPO_DIR}


Repo already exists, pulling latest...
/content/VASP
remote: Enumerating objects: 25, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 9 (delta 6), reused 8 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (9/9), 1.83 KiB | 1.83 MiB/s, done.
From https://github.com/theextraordinary/VASP
 + e2b1c82...f815f4e main       -> origin/main  (forced update)
hint: You have divergent branches and need to specify how to reconcile them.
hint: You can do so by running one of the following commands sometime before
hint: your next pull:
hint: 
hint:   git config pull.rebase false  # merge (the default strategy)
hint:   git config pull.rebase true   # rebase
hint:   git config pull.ff only       # fast-forward only
hint: 
hint: You can replace "git config" with "git config --global" to set a default
hint: preference for all repositories. You can also pass --rebase, --no-rebase,
hint: or --ff-only on the command line to override t

In [4]:
!cd /content/VASP
!git fetch origin
!git reset --hard origin/main

HEAD is now at f815f4e Tuning planner qlora


In [5]:
!pip -q install -U transformers peft bitsandbytes accelerate trl datasets

In [6]:
# 3) Install dependencies
!pip -q install datasets peft accelerate bitsandbytes trl pyyaml huggingface_hub


In [7]:
# 4) Hugging Face login (needed for google/gemma-4-e2b-it)
from huggingface_hub import login
login()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


In [9]:
# 5) Verify required files
from pathlib import Path
required = [
    "finetune/configs/train_planner_paired_qlora.yaml",
    "finetune/configs/train_refiner_paired_qlora.yaml",
    "finetune/data/planner_paired_train.jsonl",
    "finetune/data/planner_paired_val.jsonl",
    "finetune/data/refiner_paired_train.jsonl",
    "finetune/data/refiner_paired_val.jsonl",
]
missing = [f for f in required if not Path(f).exists()]
if missing:
    print("Missing files:")
    for m in missing:
        print(" -", m)
    raise SystemExit("Please upload/sync these files before training.")
print("All required files found.")


All required files found.


In [10]:
# 6) Optional sanity-check dataset counts
import json
from pathlib import Path
for f in [
    "finetune/data/planner_paired_train.jsonl",
    "finetune/data/planner_paired_val.jsonl",
    "finetune/data/refiner_paired_train.jsonl",
    "finetune/data/refiner_paired_val.jsonl",
]:
    n = sum(1 for _ in Path(f).open("r", encoding="utf-8"))
    print(f"{f}: {n}")


finetune/data/planner_paired_train.jsonl: 399
finetune/data/planner_paired_val.jsonl: 21
finetune/data/refiner_paired_train.jsonl: 399
finetune/data/refiner_paired_val.jsonl: 21


## Train Planner Adapter


In [11]:
!pip install git+https://github.com/huggingface/transformers.git

  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-0_0f0zql
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-0_0f0zql
  Resolved https://github.com/huggingface/transformers.git to commit ef97a7525dc6e4665db4ba5e1ec40d79e43cc74e
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [12]:
# 7) Train planner
!python -m finetune.training.train_qlora --config finetune/configs/train_planner_paired_qlora.yaml


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 1951/1951 [00:00<00:00, 2111.05it/s]
/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
trainable params: 59,719,680 || all params: 5,164,017,184 || trainable%: 1.1565
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/VASP/finetune/training/train_qlora.py", line 96, in <module>
    main()
  File "/content/VASP/finetune/training/train_qlora.py", line 80, in main
    trainer = Trainer(
              ^^^^^^^^
TypeError: Trainer.__init__() got an unexpected keyword argument 'tokenizer'


## Train Refiner Adapter


In [ ]:
# 8) Train refiner
!python -m finetune.training.train_qlora --config finetune/configs/train_refiner_paired_qlora.yaml


In [ ]:
# 9) (Optional) Run evaluation if you have eval config wired to new outputs
# !python -m finetune.evaluation.run_eval --config finetune/configs/eval.yaml


In [ ]:
# 10) Zip trained adapters
!zip -r planner_e2b_qlora.zip finetune/output/planner_e2b_qlora
!zip -r refiner_e2b_qlora.zip finetune/output/refiner_e2b_qlora


In [ ]:
# 11) Download to local machine
from google.colab import files
files.download("planner_e2b_qlora.zip")
files.download("refiner_e2b_qlora.zip")


In [ ]:
# 12) (Optional) Save artifacts to Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp planner_e2b_qlora.zip /content/drive/MyDrive/
# !cp refiner_e2b_qlora.zip /content/drive/MyDrive/
